In [ ]:
from pyspark import SparkConf, SparkContext
from prettytable import PrettyTable

# Instead of doing sparkContext, we can set master in the config.  
conf = SparkConf().setMaster('local[*]').setAppName('WordCount')

# Then just pass it to our context.  
sc = SparkContext(conf = conf)

# Read inn a text file.  Same directory, easy read.  
book = sc.textFile('./pride_and_prejudice.txt')

words = book.flatMap(lambda x: x.split())

# Get the dictionary of words- all unique words and the number of occurences.  
word_counts = words.countByValue()

table = PrettyTable()
table.field_names = ['Word', 'Count']


for word, count in word_counts.items():
    table.add_row([word, count])

print(table)

Improved word count collection.  Filter out other characters

In [ ]:
from pyspark import SparkConf, SparkContext
from prettytable import PrettyTable
import re

# Instead of doing sparkContext, we can set master in the config.  
conf = SparkConf().setMaster('local[*]').setAppName('WordCount')

# Then just pass it to our context.  
sc = SparkContext(conf = conf)

# Reusable regular expression.  Find beginning \b of words \w, don't capture apostrophes
WORD_RE = re.compile(r"\b\w+(?: '\w+)*\b", re.UNICODE)

def normalizeWords(line):
    return WORD_RE.findall(line.lower())

# Read inn a text file.  Same directory, easy read.  
book = sc.textFile('./pride_and_prejudice.txt')

words = book.flatMap(normalizeWords)

# Get the dictionary of words- all unique words and the number of occurences, as a tuple of the value and 1.  
word_counts = words.map(lambda x: (x, 1)).reduceByKey(lambda a, b: a + b) # Reduce by key is sort of like group by, for word a, key b in this case.  
sorted_counts = word_counts.sortBy(lambda x: x[1], ascending=False) # Sort each line x by the second column.  (The count column)

table = PrettyTable()
table.field_names = ['Word', 'Count']


for word, count in sorted_counts.collect():
    table.add_row([word, count])

print(table)

sc.stop()

This next file gets the total order value for each customer.  
Total order value by customer.  
Customer ID:  Then total of all their orders by price on the right.  Who spent the most?  

In [ ]:
from pyspark import SparkConf, SparkContext
from prettytable import PrettyTable

conf = SparkConf().setMaster('local[*]').setAppName('WordCount')

sc = SparkContext(conf = conf)


# Get all orders.  
orders = sc.textFile('./customer-orders.csv')

# Now we need to get rid of the order id for each order, since we only care about the total amount per customer.  
def filter_orders(line):
    traits = line.split(',') # CSV.  
    customer_id = traits[0]
    price = float(traits[2])
    return (customer_id, price)


# Then, for each line, we map a line to a key value RDD.  
order_pairs = orders.map(filter_orders) # This 
grouped_orders = order_pairs.reduceByKey(lambda a, b: a + b) # For each matching key, reduce two into one value as a total

sorted_orders = grouped_orders.sortBy(lambda x: x[1], ascending=False)

table = PrettyTable()
table.field_names = ['Customer ID', 'Total Spent']

for customer, total in sorted_orders.collect():
    table.add_row([customer, total])

print(table)

sc.stop()


This example is used to demonstrate the use and value of repartition.  

In [ ]:
from pyspark import SparkContext, SparkConf
from prettytable import PrettyTable
from time import time
conf = SparkConf().setMaster('local[*]').setAppName('RepartitionExample1')

sc = SparkContext(conf = conf)


def work(x):
    return x * x


DATA = 5_000

rdd = sc.parallelize(range(DATA), 2)


# force shuffle to actually happen
rdd.count()


rdd.map(work).count()


sc.stop()


In [ ]:
from pyspark import SparkContext, SparkConf
from prettytable import PrettyTable
from time import time
conf = SparkConf().setMaster('local[*]').setAppName('RepartitionExample2')

sc = SparkContext(conf = conf)


def work(x):
    return x * x


DATA = 5_000

rdd = sc.parallelize(range(DATA), 2)


rdd2 = rdd.repartition(8)

# force shuffle to actually happen
rdd2.count()

rdd2.map(work).count()

sc.stop()


This cell is used to print a dataframe of the name and age for each entry of fakefriends-header.csv


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NameAgeDataframeApp") \
    .master("local[*]") \
    .getOrCreate()

df = spark.read.csv("fakefriends-header.csv", header=True)



df.printSchema()
df.select("name", "age").show()

spark.stop()

For this cell, we'll do the same customers by order total, except using dataframes.  

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('CustomerTotals').master('local[*]').getOrCreate()

df = spark.read.csv("./customer-orders.csv", schema="customerID STRING, orderId STRING, orderPrice DOUBLE")

df.createOrReplaceTempView('orders')

customerTotal = spark.sql("""SELECT customerId, SUM(orderPrice) AS TOTAL FROM orders GROUP BY customerId ORDER BY TOTAL DESC""")

customerTotal.show()

spark.stop()

In [ ]:
from pyspark.sql import SparkSession, Row
spark = SparkSession.builder.appName('FriendsOverAge').master('local[*]').getOrCreate()

# Get the id, nname, age, and number of friends as a row.  
def parseLine(line):
    fields = line.split(',')
    return Row(
    ID=int(fields[0]),
    name=str(fields[1].encode('utf-8')),
    age=int(fields[2]),
    num_friends=int(fields[3])
)

lines = spark.sparkContext.textFile("./fakefriends.csv")

# Row.map returns a dataframe.  
people = lines.map(parseLine)

# We cache this because we'll be using it multiple times.  
peopleDf = spark.createDataFrame(people).cache()

peopleDf.createOrReplaceTempView('people')

adults = spark.sql("SELECT * FROM people WHERE AGE >= 18")

for p in adults.collect():
    print(p)

peopleDf.groupBy('age').count().orderBy('age').show()


spark.stop()

We used sql to get some of these things, then we use dataframe filtering and groupby.  
What's the preferred method?  Does it matter?  

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('AnotherAgeExample').getOrCreate()

# Infer schema lets us automatically detect types.  Otherwise they default to string.  
people = spark.read.option('header', True).option('inferschema', True).csv('./fakefriends-header.csv')

people.select('name').show()

people.filter(people.age <= 35).show()

people.groupBy('age').count().show()

people.select(people.name, people.age + 10).show()

spark.stop()

+--------+
|    name|
+--------+
|    Will|
|Jean-Luc|
|    Hugh|
|  Deanna|
|   Quark|
|  Weyoun|
|  Gowron|
|    Will|
|  Jadzia|
|    Hugh|
|     Odo|
|     Ben|
|   Keiko|
|Jean-Luc|
|    Hugh|
|     Rom|
|  Weyoun|
|     Odo|
|Jean-Luc|
|  Geordi|
+--------+
only showing top 20 rows
+------+--------+---+-------+
|userID|    name|age|friends|
+------+--------+---+-------+
|     0|    Will| 33|    385|
|     1|Jean-Luc| 26|      2|
|     9|    Hugh| 27|    181|
|    16|  Weyoun| 22|    323|
|    17|     Odo| 35|     13|
|    21|   Miles| 19|    268|
|    22|   Quark| 30|     72|
|    24|  Julian| 25|      1|
|    25|     Ben| 21|    445|
|    26|  Julian| 22|    100|
|    32|     Nog| 26|    281|
|    35| Beverly| 27|    305|
|    36|  Kasidy| 32|     81|
|    39|    Morn| 31|    192|
|    44|   Nerys| 35|    244|
|    46|    Morn| 25|     96|
|    47|   Brunt| 24|     49|
|    48|     Nog| 20|      1|
|    52| Beverly| 19|    269|
|    54|   Brunt| 19|      5|
+------+--------+---+